### ENTREGA

In [2]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import time

In [10]:
# --------------------------------------------------
# RUTAS DE LOS ACCESORIOS
# --------------------------------------------------

carpeta = Path.cwd() / "accesorios"

ruta_sombrero = carpeta / "sombrero.png"
ruta_puro = carpeta / "puro.png"
ruta_bandana = carpeta / "bandadna.png"

# --------------------------------------------------
# CARGA DE IMAGENES
# --------------------------------------------------

def cargar_accesorio(ruta):
    imagen = cv2.imread(str(ruta), cv2.IMREAD_UNCHANGED)

    if imagen is None:
        print(f"No se ha encontrado: {ruta.name}")
        return None

    # Si la imagen no tiene transparencia, añadimos un canal alfa.
    if len(imagen.shape) == 3 and imagen.shape[2] == 3:
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2BGRA)

    return imagen


sombrero = cargar_accesorio(ruta_sombrero)
puro = cargar_accesorio(ruta_puro)
bandana = cargar_accesorio(ruta_bandana)

# --------------------------------------------------
# SUPERPOSICION DE IMAGENES
# --------------------------------------------------

def superponer(imagen, accesorio, centro, ancho, angulo=0):

    if accesorio is None:
        return imagen

    alto_original, ancho_original = accesorio.shape[:2]

    # Mantener la proporcion original del accesorio.
    alto = int(alto_original * ancho / ancho_original)

    if ancho <= 0 or alto <= 0:
        return imagen

    accesorio = cv2.resize(accesorio, (ancho, alto))

    # Girar el accesorio para adaptarlo a la inclinacion de la cabeza.
    matriz = cv2.getRotationMatrix2D(
        (ancho // 2, alto // 2), angulo, 1
    )

    accesorio = cv2.warpAffine(
        accesorio,
        matriz,
        (ancho, alto),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    alto_img, ancho_img = imagen.shape[:2]

    # Coordenadas de la esquina superior izquierda.
    x = int(centro[0] - ancho / 2)
    y = int(centro[1] - alto / 2)

    # Recortar si el accesorio sale de los limites de la imagen.
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(ancho_img, x + ancho)
    y2 = min(alto_img, y + alto)

    if x1 >= x2 or y1 >= y2:
        return imagen

    # Recortar tambien la imagen del accesorio.
    accesorio_recortado = accesorio[
        y1 - y:y2 - y,
        x1 - x:x2 - x
    ]

    # Separar los canales de color y transparencia.
    zona = imagen[y1:y2, x1:x2]

    alfa = accesorio_recortado[:, :, 3] / 255.0
    alfa = alfa[:, :, np.newaxis]

    colores = accesorio_recortado[:, :, :3]

    # Mezclar el accesorio con la imagen original.
    imagen[y1:y2, x1:x2] = (
        colores * alfa + zona * (1 - alfa)
    ).astype(np.uint8)

    return imagen


# --------------------------------------------------
# DETECCION FACIAL
# --------------------------------------------------

modelo = Path.cwd() / "face_landmarker.task"

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

opciones = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(modelo)),
    running_mode=RunningMode.VIDEO,
    num_faces=1
)

face_landmarker = FaceLandmarker.create_from_options(opciones)


# --------------------------------------------------
# CAPTURA DE VIDEO
# --------------------------------------------------

camara = cv2.VideoCapture(0)

if not camara.isOpened():
    print("No se ha podido acceder a la camara.")
    exit()

ultimo_timestamp = -1

while True:

    correcto, frame = camara.read()

    if not correcto:
        break

    # Reflejar la imagen como si fuera un espejo.
    frame = cv2.flip(frame, 1)

    alto, ancho = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convertir la imagen al formato de MediaPipe.
    mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb
    )

# Obtener una marca de tiempo creciente.
    timestamp_ms = max(
    int(time.monotonic() * 1000),
    ultimo_timestamp + 1
    )
    ultimo_timestamp = timestamp_ms

# Detectar los puntos faciales.
    resultados = face_landmarker.detect_for_video(
    mp_image,
    timestamp_ms
    )

    if resultados.face_landmarks:

        cara = resultados.face_landmarks[0]

        # Obtener las coordenadas de un punto facial.
        def punto(indice):
            landmark = cara[indice]

            x = int(landmark.x * ancho)
            y = int(landmark.y * alto)

            return (x, y)

        # Puntos principales de la cara.
        frente = punto(6)
        barbilla = punto(152)

        lado_izquierdo = punto(234)
        lado_derecho = punto(454)

        comisura_izquierda = punto(88)
        comisura_derecha = punto(291)

        boca_superior = punto(13)

        # Anchura aproximada de la cara.
        ancho_cara = int(
            np.linalg.norm(
                np.array(lado_derecho) -
                np.array(lado_izquierdo)
            )
        )

        # Inclinacion de la cabeza.
        vector_cara = (
            np.array(lado_derecho) -
            np.array(lado_izquierdo)
        )

        angulo = np.degrees(
            np.arctan2(vector_cara[1], vector_cara[0])
        )

        # ------------------------------------------
        # SOMBRERO
        # ------------------------------------------

        if sombrero is not None:

            ancho_sombrero = int(ancho_cara * 2.2)

            alto_sombrero = int(
                sombrero.shape[0] *
                ancho_sombrero /
                sombrero.shape[1]
            )

            centro_sombrero = (
                frente[0],
                frente[1] - int(alto_sombrero * 0.45)
            )

            frame = superponer(
                frame,
                sombrero,
                centro_sombrero,
                ancho_sombrero,
                angulo
            )

        # ------------------------------------------
        # PURO
        # ------------------------------------------

        if puro is not None:

            ancho_puro = int(ancho_cara * 0.40)

            centro_puro = (
                comisura_derecha[0] + int(ancho_cara * 0.13),
                boca_superior[1] + int(ancho_cara * 0.03)
            )

            frame = superponer(
                frame,
                puro,
                centro_puro,
                ancho_puro,
                angulo
            )

        # ------------------------------------------
        # BANDANA
        # ------------------------------------------

        if bandana is not None:

            ancho_bandana = int(ancho_cara * 1.30)

            centro_bandana = (
                barbilla[0],
                barbilla[1] + int(ancho_cara * 0.40)
            )

            frame = superponer(
                frame,
                bandana,
                centro_bandana,
                ancho_bandana,
                angulo
            )

    # Mostrar resultado.
    cv2.imshow("Filtro de vaquero", frame)

    # Salir pulsando ESC.
    if cv2.waitKey(1) & 0xFF == 27:
        break


# --------------------------------------------------
# LIBERAR RECURSOS
# --------------------------------------------------

camara.release()
face_landmarker.close()
cv2.destroyAllWindows()
